In [1]:
from bs4 import BeautifulSoup
import requests
import cloudscraper
import time
import random
import re

In [2]:
scraper = cloudscraper.create_scraper()

In [3]:
def get_number_of_weeks(season: int) -> int | None:
    """Return the number of weeks in a season."""
    week = 1
    while True:
        url = f"https://www.pro-football-reference.com/years/{season}/week_{week}.htm"
        response = scraper.get(url)
        # Stop if the page does not exist (404 or other errors)
        if response.status_code != 200:
            break
        # Stop if the page is a preview for a future week
        title = response.text.split("<title>")[1].split("</title>")[0]
        if "Preview" in title:
            break
        week += 1
        time.sleep(random.uniform(1, 3))
    completed_weeks = week - 1
    return completed_weeks if completed_weeks > 0 else None

In [4]:
def get_week_links(season: int, week: int) -> list[str]:
    """Get the links to all games in a week."""
    week_links = []
    url = f"https://www.pro-football-reference.com/years/{season}/week_{week}.htm"
    response = scraper.get(url)
    soup = BeautifulSoup(response.text, "lxml")
    soup_links = soup.find_all("td", class_="gamelink")
    for l in soup_links:
        link = l.a["href"]
        full_link = f"https://www.pro-football-reference.com{link}"
        week_links.append(full_link)
    return week_links

In [5]:
def get_season_links(season: int):
    """Get the links to all games in a season."""
    season_links = []
    number_of_weeks = get_number_of_weeks(season)
    if number_of_weeks is None:
        return season_links
    for week in range(1, number_of_weeks + 1):
        week_links = get_week_links(season, week)
        season_links.extend(week_links)
        time.sleep(random.uniform(1, 3))
    return season_links

In [6]:
test_links = get_season_links(2010)
len(test_links) #Should be 267 games

267

In [3]:
url = "https://www.pro-football-reference.com/boxscores/201009090nor.htm"
response = scraper.get(url).text
clean_response = re.sub(r"<!--|-->", "", response)
soup = BeautifulSoup(clean_response, "lxml")

In [4]:
def extract_game_info(soup: BeautifulSoup) -> dict:
    """
    Extracts key game information from a single NFL game page.

    Args:
        soup: Parsed HTML of the game page.

    Returns:
        dict: Dictionary containing home and away teams, game metadata
              (roof, surface, weather, attendance), home line, and over/under.
    """
    # Initialize dictionary with all expected keys as None
    game_info = {
        "home_team": None,
        "away_team": None,
        "roof": None,
        "surface": None,
        "weather": None,
        "attendance": None,
        "home_line": None,
        "over_under": None
    }

    # Extract home and away teams from scoring table
    scorebox = soup.find("table", id="scoring")
    game_info["home_team"] = scorebox.find("th", {"data-stat": "home_team_score"}).text.strip()
    game_info["away_team"] = scorebox.find("th", {"data-stat": "vis_team_score"}).text.strip()

    # Extract game metadata and betting lines from game info table
    table = soup.find("table", id="game_info")
    for row in table.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue
        field_name = th.text.strip().lower()
        if field_name == "roof":
            game_info["roof"] = td.text
        elif field_name == "surface":
            game_info["surface"] = td.text
        elif field_name == "weather":
            game_info["weather"] = td.text
        elif field_name == "attendance":
            game_info["attendance"] = int(td.text.replace(",", ""))
        elif field_name == "vegas line":
            away_team_full_name = soup.title.string.split(" at ")[0]
            home_team_full_name = soup.title.string.split(" at ")[1].split(" - ")[0]
            vegas_favorite = td.text.split()[0].strip()
            vegas_line = float(td.text.split()[-1])
            if vegas_favorite in home_team_full_name:
                game_info["home_line"] = vegas_line
            elif vegas_favorite in away_team_full_name:
                game_info["home_line"] = -vegas_line
            else:
                raise ValueError("Vegas favorite does not match either team.")
        elif field_name == "over/under":
            game_info["over_under"] = float(td.text.split()[0])

    return game_info
                

In [5]:
tst = extract_game_info(soup)
tst

{'home_team': 'NOR',
 'away_team': 'MIN',
 'roof': 'dome',
 'surface': 'sportturf',
 'weather': None,
 'attendance': 70051,
 'home_line': -5.0,
 'over_under': 49.5}